# Evaluate diffusion samples on CPU

Evaluate the same 20 geometries and observable targets as `transformer/evaluate.ipynb`.
`diffusion/evaluate.py` imports `convert_samples_to_data`, `validate_vacua`, and `print_console_output` from `transformer/evaluation_scripts/`. It uses the same geometry log, solver settings and transformer training data for novelty checks. Evaluation does not load either model's neural-network checkpoints.

Inputs: `diffusion/samples/diffusion_{mode}_samples.p` and optional matching `.json` sampling metadata. In a separate Colab runtime, upload/copy the saved sample files into this directory before evaluating; opening this notebook does not transfer outputs from the sampling runtime.

Outputs: `diffusion/evaluations/*_valid.p` and `*_summary.json`. Existing result files are protected from overwriting.


## Install software (CPU runtime)


In [ ]:
%cd /content
!git clone https://github.com/mowalden/towards_type_IIB_foundation_model.git
%cd towards_type_IIB_foundation_model
%pip install -r requirements.txt
%cd diffusion

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_ROOT = Path("/content/towards_type_IIB_foundation_model")

## Evaluation settings

`DISCR = None` reads the scale from each sample's JSON sidecar, falling back to 10 when no sidecar is present. A supplied scale must agree with that metadata. For `gs_inv`, the same transformer convention sets the initial Im(tau) to the prompt value 4. Sample counts are read directly from the files.

The novelty counts below are relative to the transformer's reference training datasets, matching its evaluation notebook.


In [ ]:
SAMPLES_DIR = REPO_ROOT / "diffusion/samples"
RESULTS_DIR = REPO_ROOT / "diffusion/evaluations"
DISCR = None
TOL = 1e-10
USE_NEWTON = False  # Default: the transformer's SciPy root solver.
VERBOSE = False


def evaluate(mode, filename=None):
    sample_path = SAMPLES_DIR / (filename or f"diffusion_{mode}_samples.p")
    if not sample_path.is_file():
        raise FileNotFoundError(f"Copy your saved diffusion samples to {sample_path}")
    command = [
        sys.executable, "-u", str(REPO_ROOT / "diffusion/evaluate.py"),
        "--mode", mode, "--samples_file", str(sample_path),
        "--output_dir", str(RESULTS_DIR), "--tol", str(TOL),
    ]
    if DISCR is not None:
        command += ["--discr", str(DISCR)]
    if USE_NEWTON:
        command.append("--use_newton")
    if VERBOSE:
        command.append("--verbose")
    subprocess.run(command, check=True)


## Evaluate $N_{\rm flux}=40$


In [ ]:
evaluate("nflux")


## Evaluate $|W_0|=1.1$


In [ ]:
evaluate("w0")


## Evaluate $1/g_s=4$


In [ ]:
evaluate("gs_inv")


## Evaluate unconditional samples (optional)

Each sample is evaluated on its prompted geometry, corresponding to `--mod --single` in the transformer evaluator.


In [ ]:
evaluate("unconditional")


## Saved results

Conditional `*_valid.p` files use the transformer's `{prompt: {"dist": ..., "validity": ...}}` structure. Distributions contain unique valid solutions; the `gs_inv` distribution is Im(tau), as in the transformer evaluator. The unconditional pickle contains per-geometry counts.

Each `*_summary.json` records per-geometry sample/validity/novelty counts, aggregate rates, solver settings, coordinate scale and any available sampling metadata. To rerun without overwriting, choose another `RESULTS_DIR`.

Command-line equivalent from the repository root:

```bash
python diffusion/evaluate.py --mode nflux
```
